In [1]:
%matplotlib tk

In [2]:
import numpy as np
import tifffile
import glob
import os
from skimage import filters, morphology, exposure
from scipy import ndimage
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import porespy as ps
import openpnm as op
from scipy import stats

In [5]:
# Segmentation functions (definitions only)

def extract_pixel_features(img):
    img_norm = (img.astype(np.float64) - img.min()) / (img.max() - img.min())
    img_eq = exposure.equalize_adapthist(img_norm)
    features = [img_eq]
    for sigma in [1, 2, 4, 8]:
        features.append(filters.gaussian(img_eq, sigma=sigma))
    features.append(filters.sobel(img_eq))
    for size in [3, 7, 15]:
        local_mean = ndimage.uniform_filter(img_eq, size=size)
        local_sq_mean = ndimage.uniform_filter(img_eq**2, size=size)
        local_var = np.maximum(local_sq_mean - local_mean**2, 0)
        features.append(local_var)
    return np.stack(features, axis=-1)


def label_pixels_interactively(img, class_names=("pore", "kerogen", "mineral"), points_per_class=15):
    coords = []
    labels = []
    for class_id, name in enumerate(class_names):
        plt.figure(figsize=(10, 7))
        plt.imshow(img, cmap="gray")
        plt.title(f"Click {points_per_class} points on clear examples of: {name.upper()}\n(close window when done with this class)")
        pts = plt.ginput(n=points_per_class, timeout=0)
        plt.close()
        for (x, y) in pts:
            coords.append((int(round(y)), int(round(x))))
            labels.append(class_id)
    return np.array(coords), np.array(labels)

In [6]:
# Run segmentation on all 20 images

folder_path = "C:/FIB_SEM Images of sample A"
training_image_filenames = [
    "Image - SliceImage - 001.tif",
    "Image - SliceImage - 003.tif",
    "Image - SliceImage - 007.tif",
    "Image - SliceImage - 009.tif",
    "Image - SliceImage - 012.tif",
    "Image - SliceImage - 017.tif",
    "Image - SliceImage - 020.tif"
]

X_train_all = []
y_train_all = []

for fname in training_image_filenames:
    print(f"Labeling on {fname}...")
    img = tifffile.imread(os.path.join(folder_path, fname))
    coords, labels = label_pixels_interactively(img, points_per_class=15)
    feature_stack = extract_pixel_features(img)
    X_train_all.append(feature_stack[coords[:, 0], coords[:, 1], :])
    y_train_all.append(labels)

X_train = np.concatenate(X_train_all)
y_train = np.concatenate(y_train_all)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
print("Classifier trained on 5 images.")

all_files = sorted(glob.glob(os.path.join(folder_path, "*.tif")))
print(f"Found {len(all_files)} .tif files, segmenting all automatically...")

segmented_images = {}
porosities = {}

for fpath in all_files:
    fname = os.path.basename(fpath)
    img = tifffile.imread(fpath)
    feats = extract_pixel_features(img)
    H, W, n_feat = feats.shape
    predicted = clf.predict(feats.reshape(-1, n_feat)).reshape(H, W)
    segmented_images[fname] = predicted
    porosities[fname] = (predicted == 0).mean()
    print(f"  {fname}: porosity = {porosities[fname]:.4f}")

print()
print(f"Mean porosity across {len(all_files)} slices: {np.mean(list(porosities.values())):.4f}")

Labeling on Image - SliceImage - 001.tif...
Labeling on Image - SliceImage - 003.tif...
Labeling on Image - SliceImage - 007.tif...
Labeling on Image - SliceImage - 009.tif...
Labeling on Image - SliceImage - 012.tif...
Labeling on Image - SliceImage - 017.tif...
Labeling on Image - SliceImage - 020.tif...
Classifier trained on 5 images.
Found 20 .tif files, segmenting all automatically...
  Image - SliceImage - 001.tif: porosity = 0.0342
  Image - SliceImage - 002.tif: porosity = 0.0377
  Image - SliceImage - 003.tif: porosity = 0.0390
  Image - SliceImage - 004.tif: porosity = 0.0433
  Image - SliceImage - 005.tif: porosity = 0.0417
  Image - SliceImage - 006.tif: porosity = 0.0488
  Image - SliceImage - 007.tif: porosity = 0.0576
  Image - SliceImage - 008.tif: porosity = 0.0705
  Image - SliceImage - 009.tif: porosity = 0.0809
  Image - SliceImage - 010.tif: porosity = 0.0922
  Image - SliceImage - 011.tif: porosity = 0.1096
  Image - SliceImage - 012.tif: porosity = 0.1047
  Image

In [7]:
import matplotlib.pyplot as plt

training_img = tifffile.imread(os.path.join(folder_path, "Image - SliceImage - 012.tif"))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(training_img, cmap="gray")
axes[0].set_title("Original training image")
axes[1].imshow(segmented_images["Image - SliceImage - 012.tif"], cmap="viridis")
axes[1].set_title("Segmented (dark=pore, mid=kerogen, bright=mineral)")
plt.show()

In [8]:
# Extract pore-network statistics, with the fracture-fix (sigma) and 2D-to-3D correction

def extract_slice_statistics(labels, voxel_size_nm=5.0, sigma=0.6):
    
    pore_phase = (labels == 0)
    snow_output = ps.networks.snow2(pore_phase, voxel_size=voxel_size_nm * 1e-9, sigma=sigma)
    pn = op.io.network_from_porespy(snow_output.network)

    throat_radii = pn["throat.inscribed_diameter"] / 2   # meters (this library version names it diameter, not radius)
    coord_numbers = np.bincount(
        np.concatenate([pn["throat.conns"][:, 0], pn["throat.conns"][:, 1]])
    )  # how many throats touch each pore
    porosity = pore_phase.mean()

    return throat_radii, coord_numbers, porosity


def correct_2d_coordination_to_3d(measured_coord_2d, correction_factor=2.5):
   
    return measured_coord_2d * correction_factor


all_radii = []
all_coords = []
for fname, labels in segmented_images.items():
    radii, coord_numbers, porosity = extract_slice_statistics(labels)
    all_radii.append(radii)
    all_coords.append(coord_numbers.mean())

pooled_radii = np.concatenate(all_radii)
mean_coord_raw = np.mean(all_coords)
mean_coord = correct_2d_coordination_to_3d(mean_coord_raw)
print(f'Pooled {len(pooled_radii)} throat radii across all slices')
print(f'Raw mean coordination number: {mean_coord_raw:.2f}')
print(f'Corrected mean coordination number: {mean_coord:.2f}')

Pooled 14333 throat radii across all slices
Raw mean coordination number: 1.18
Corrected mean coordination number: 2.94


In [27]:
# Network building and Knudsen-corrected flow physics with REAL reservoir T/P from Fu et al. 2019

def build_statistical_network(pooled_radii, mean_coord_number, n_pores_per_side=15, max_attempts=30):
   
    from scipy.sparse import coo_matrix
    from scipy.sparse.csgraph import connected_components

    for attempt in range(max_attempts):
        net = op.network.Cubic(shape=[n_pores_per_side]*3, spacing=50e-9)

        target_throats = int(net.Nt * mean_coord_number / 6.0)
        n_to_trim = net.Nt - target_throats
        if n_to_trim > 0:
            trim_ids = np.random.choice(net.Ts, size=n_to_trim, replace=False)
            op.topotools.trim(net, throats=trim_ids)

        # Find ALL connected components, then physically remove every
        # pore that isn't part of the SINGLE LARGEST one. 
        # Actually removing those strays guarantees what's left is
        # one single, fully connected network, no exceptions.
        conns = net["throat.conns"]
        rows = np.concatenate([conns[:, 0], conns[:, 1]])
        cols = np.concatenate([conns[:, 1], conns[:, 0]])
        data = np.ones(len(rows))
        adjacency = coo_matrix((data, (rows, cols)), shape=(net.Np, net.Np))
        n_components, component_labels = connected_components(adjacency, directed=False)

        largest_label = np.bincount(component_labels).argmax()
        stray_pores = np.where(component_labels != largest_label)[0]
        if len(stray_pores) > 0:
            op.topotools.trim(net, pores=stray_pores)

        # After removing strays, confirm the remaining single-piece
        # network still actually reaches both the inlet and outlet
        
        if net.pores("left").size > 0 and net.pores("right").size > 0:
            break  # genuinely percolating network, stop retrying
    else:
        raise RuntimeError(
            f"Could not generate a percolating network after {max_attempts} "
            f"attempts at mean_coord={mean_coord_number:.2f}. This coordination "
            f"number may be too low (close to or below the percolation "
            f"threshold) for a {n_pores_per_side}^3 cubic lattice -- consider "
            f"a larger n_pores_per_side, or double-check mean_coord isn't "
            f"artificially low due to a segmentation issue upstream."
        )

    # Fit a log-normal distribution to the pooled throat radii.
 
    shape, loc, scale = stats.lognorm.fit(pooled_radii, floc=0)
    sampled_radii = stats.lognorm.rvs(shape, loc=loc, scale=scale, size=net.Nt)
    net["throat.radius"] = sampled_radii
    net["throat.diameter"] = 2 * sampled_radii

    # Pore radii: approximate as the max of their connected throat
    # radii scaled up slightly 
    pore_radii = np.zeros(net.Np)
    for p in range(net.Np):
        connected = net.find_neighbor_throats(pores=p)
        if len(connected) > 0:
            pore_radii[p] = np.max(sampled_radii[connected]) * 1.3
        else:
            pore_radii[p] = np.median(sampled_radii)
    net["pore.diameter"] = 2 * pore_radii

    return net


#  Knudsen-corrected hydraulic conductance

K_B = 1.380649e-23
T_reservoir = 398.15    # K (125 C / 256 F), from Fu et al. 2019 (URTeC 402), Sample A reservoir condition
P_reservoir = 2.413e7    # Pa (3500 psi), from Fu et al. 2019 (URTeC 402), Sample A reservoir condition
d_molecule_CH4 = 3.8e-10
alpha_rarefaction = 0.8
mu_CH4 = 1.1e-5          # Pa.s, methane viscosity at reservoir conditions

def mean_free_path(T, P, d_molecule):
    return (K_B * T) / (np.sqrt(2) * np.pi * d_molecule**2 * P)

def knudsen_correction(Kn, alpha=alpha_rarefaction):
    return (1.0 + alpha * Kn) * (1.0 + 4.0 * Kn / (1.0 + Kn))

def assign_conductance(net, mu=mu_CH4):
  
    lam = mean_free_path(T_reservoir, P_reservoir, d_molecule_CH4)
    r = net["throat.radius"]
    L = net["throat.length"] if "throat.length" in net.keys() else net["throat.diameter"]

    Kn = lam / (2 * r)
    correction = knudsen_correction(Kn)

    g_continuum = (np.pi * r**4) / (8 * mu * L)
    g_apparent = g_continuum * correction

    net["throat.conductance"] = g_apparent
    net["throat.conductance_intrinsic"] = g_continuum   # kept for controlled comparison, not discarded
    net["throat.Kn"] = Kn
    return net

# Single-phase Stokes flow -> apparent permeability

def run_stokes_flow(net):
    
    phase = op.phase.Phase(network=net)
    phase["throat.hydraulic_conductance"] = net["throat.conductance"]

    inlet = net.pores("left")
    outlet = net.pores("right")

    flow = op.algorithms.StokesFlow(network=net, phase=phase)
    flow.set_value_BC(pores=inlet, values=101325 + 1000)   # Pa, +dP inlet
    flow.set_value_BC(pores=outlet, values=101325)          # Pa, outlet
    flow.run()

    Q = flow.rate(pores=inlet)[0]                # m^3/s
    dP = 1000                                     # Pa
    L = net["pore.coords"][:, 0].max() - net["pore.coords"][:, 0].min()
    A = (net["pore.coords"][:, 1].max() - net["pore.coords"][:, 1].min()) * \
        (net["pore.coords"][:, 2].max() - net["pore.coords"][:, 2].min())

    k_apparent = (Q * mu_CH4 * L) / (A * dP)      # Darcy's law, m^2
    return k_apparent, flow


def run_stokes_flow_intrinsic(net):
   
    phase = op.phase.Phase(network=net)
    phase["throat.hydraulic_conductance"] = net["throat.conductance_intrinsic"]

    inlet = net.pores("left")
    outlet = net.pores("right")

    flow = op.algorithms.StokesFlow(network=net, phase=phase)
    flow.set_value_BC(pores=inlet, values=101325 + 1000)
    flow.set_value_BC(pores=outlet, values=101325)
    flow.run()

    Q = flow.rate(pores=inlet)[0]
    dP = 1000
    L = net["pore.coords"][:, 0].max() - net["pore.coords"][:, 0].min()
    A = (net["pore.coords"][:, 1].max() - net["pore.coords"][:, 1].min()) * \
        (net["pore.coords"][:, 2].max() - net["pore.coords"][:, 2].min())

    k_intrinsic = (Q * mu_CH4 * L) / (A * dP)
    return k_intrinsic, flow


In [28]:
# Define the CO2-EOR displacement function 

def co2_eor_invasion(net, langmuir_qmax_CH4=0.5, langmuir_b_CH4=1e-7,
                      langmuir_qmax_CO2=0.9, langmuir_b_CO2=3e-7,
                      P=P_reservoir):
   
    def adsorbed_thickness(q_max, b, P, molecule_diameter):
        theta = (b * P) / (1 + b * P)     # Langmuir fractional coverage
        return theta * molecule_diameter   # monolayer approx.

    d_CH4 = 3.8e-10
    d_CO2 = 3.3e-10

    t_CH4 = adsorbed_thickness(langmuir_qmax_CH4, langmuir_b_CH4, P, d_CH4)
    t_CO2 = adsorbed_thickness(langmuir_qmax_CO2, langmuir_b_CO2, P, d_CO2)

    # Effective radius for CO2 invasion = original radius minus the
    # CO2 adsorbed layer it must displace CH4's own adsorbed layer to
    # form -- simplified as net additional shrinkage from CO2's
    # stronger uptake relative to CH4's baseline.
    net["throat.radius_CO2"] = net["throat.radius"] - max(t_CO2 - t_CH4, 0)
    net["throat.diameter_CO2"] = 2 * net["throat.radius_CO2"]

    water = op.phase.Phase(network=net)  # standing in for CH4 (defending)
    co2 = op.phase.Phase(network=net)    # invading

    sigma = 0.03   # N/m, CO2-CH4 interfacial tension -- LITERATURE ESTIMATE
    theta = 30     # degrees, contact angle on kerogen -- LITERATURE ESTIMATE
    net["throat.entry_pressure"] = (2 * sigma * np.cos(np.radians(theta))) / \
                                     net["throat.radius_CO2"]

    ip = op.algorithms.InvasionPercolation(network=net, phase=co2)
    ip.set_inlet_BC(pores=net.pores("left"))
    ip.run()

    outlet_pores = net.pores("right")
    outlet_sequences = ip["pore.invasion_sequence"][outlet_pores]
    finite_outlet_sequences = outlet_sequences[np.isfinite(outlet_sequences)]

    if len(finite_outlet_sequences) == 0:
        # CO2 never reached the outlet at all -- no breakthrough,
        # recovery factor should reflect that (near zero), not
        # whatever partial invasion happened to occur.
        recovery_factor = 0.0
    else:
        breakthrough_sequence = finite_outlet_sequences.min()
        invaded_at_breakthrough = ip["pore.invasion_sequence"] <= breakthrough_sequence
        recovery_factor = invaded_at_breakthrough.sum() / net.Np

    return recovery_factor, ip

In [29]:
# Run 200 Monte Carlo realizations, get P10/P50/P90
# both k_intrinsic and k_apparent for the 3-way comparison

def run_single_realization(pooled_radii, mean_coord, seed):
    np.random.seed(seed)
    net = build_statistical_network(pooled_radii, mean_coord)
    net = assign_conductance(net)
    k_intrinsic, _ = run_stokes_flow_intrinsic(net)
    k_apparent, _ = run_stokes_flow(net)
    recovery_factor, _ = co2_eor_invasion(net)
    return k_intrinsic, k_apparent, recovery_factor

n_realizations = 200
k_intrinsic_results = []
k_apparent_results = []
rf_results = []

for i in range(n_realizations):
    k_int, k_app, rf = run_single_realization(pooled_radii, mean_coord, seed=i)
    k_intrinsic_results.append(k_int)
    k_apparent_results.append(k_app)
    rf_results.append(rf)

k_intrinsic_results = np.array(k_intrinsic_results)
k_apparent_results = np.array(k_apparent_results)
rf_results = np.array(rf_results)

def p10_p50_p90(arr):
    return np.percentile(arr, 90), np.percentile(arr, 50), np.percentile(arr, 10)

ki_p10, ki_p50, ki_p90 = p10_p50_p90(k_intrinsic_results)
ka_p10, ka_p50, ka_p90 = p10_p50_p90(k_apparent_results)
rf_p10, rf_p50, rf_p90 = p10_p50_p90(rf_results)

print(f"Ran {n_realizations} realizations")
print()
print("Intrinsic permeability, continuum physics (m^2):")
print(f"  P90 (conservative): {ki_p90:.3e}")
print(f"  P50 (most likely):  {ki_p50:.3e}")
print(f"  P10 (optimistic):   {ki_p10:.3e}")
print()
print("Apparent permeability, Knudsen-corrected (m^2):")
print(f"  P90 (conservative): {ka_p90:.3e}")
print(f"  P50 (most likely):  {ka_p50:.3e}")
print(f"  P10 (optimistic):   {ka_p10:.3e}")
print()
print(f"Physics-only correction ratio (P50 apparent / P50 intrinsic): {ka_p50/ki_p50:.3f}x")
print()
print("CO2-EOR recovery factor (at breakthrough):")
print(f"  P90 (conservative): {rf_p90:.3f}")
print(f"  P50 (most likely):  {rf_p50:.3f}")
print(f"  P10 (optimistic):   {rf_p10:.3f}")

Ran 200 realizations

Intrinsic permeability, continuum physics (m^2):
  P90 (conservative): 6.193e-19
  P50 (most likely):  6.764e-19
  P10 (optimistic):   7.475e-19

Apparent permeability, Knudsen-corrected (m^2):
  P90 (conservative): 6.904e-19
  P50 (most likely):  7.510e-19
  P10 (optimistic):   8.284e-19

Physics-only correction ratio (P50 apparent / P50 intrinsic): 1.110x

CO2-EOR recovery factor (at breakthrough):
  P90 (conservative): 0.187
  P50 (most likely):  0.254
  P10 (optimistic):   0.339


In [30]:
# Train the ML surrogate model, see which property matters most

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import shortest_path

def run_realization_with_features(pooled_radii, mean_coord, seed):
    np.random.seed(seed)
    net = build_statistical_network(pooled_radii, mean_coord)
    net = assign_conductance(net)

  
    conns = net["throat.conns"]
    weights = 1.0 / (net["throat.conductance"] + 1e-30)  # higher conductance = "shorter" effective path
    rows = np.concatenate([conns[:, 0], conns[:, 1]])
    cols = np.concatenate([conns[:, 1], conns[:, 0]])
    data = np.concatenate([weights, weights])
    graph = coo_matrix((data, (rows, cols)), shape=(net.Np, net.Np))

    inlet_pores = net.pores("left")
    outlet_pores = net.pores("right")
    dist_matrix = shortest_path(graph, indices=inlet_pores[0], directed=False)
    finite_outlet_dists = dist_matrix[outlet_pores][np.isfinite(dist_matrix[outlet_pores])]
    shortest_path_length = finite_outlet_dists.min() if len(finite_outlet_dists) > 0 else np.nan

    features = {
        "mean_throat_radius": np.mean(net["throat.radius"]),
        "std_throat_radius": np.std(net["throat.radius"]),
        "mean_Kn": np.mean(net["throat.Kn"]),
        "n_throats": net.Nt,
        "shortest_path_length": shortest_path_length,
    }
    k_apparent, _ = run_stokes_flow(net)
    recovery_factor, _ = co2_eor_invasion(net)
    return features, k_apparent, recovery_factor

X, y_k, y_rf = [], [], []
for i in range(200):
    feats, k, rf = run_realization_with_features(pooled_radii, mean_coord, seed=i)
    X.append(list(feats.values()))
    y_k.append(k)
    y_rf.append(rf)

feature_names = list(feats.keys())
X = np.array(X)
y_k = np.array(y_k)
y_rf = np.array(y_rf)

def train_surrogate(X, y, feature_names, target_name):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    from sklearn.ensemble import RandomForestRegressor
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    print(f"\nSurrogate model for {target_name}:")
    print(f"  R^2 on held-out test set: {r2:.3f}")
    print(f"  Feature importance:")
    for name, importance in sorted(zip(feature_names, model.feature_importances_), key=lambda x: -x[1]):
        print(f"    {name}: {importance:.3f}")
    return model, r2

train_surrogate(X, y_k, feature_names, "apparent permeability")
train_surrogate(X, y_rf, feature_names, "CO2-EOR recovery factor")


Surrogate model for apparent permeability:
  R^2 on held-out test set: -0.113
  Feature importance:
    mean_throat_radius: 0.000
    std_throat_radius: 0.000
    mean_Kn: 0.000
    n_throats: 0.000
    shortest_path_length: 0.000

Surrogate model for CO2-EOR recovery factor:
  R^2 on held-out test set: -0.328
  Feature importance:
    shortest_path_length: 0.390
    mean_Kn: 0.385
    n_throats: 0.226
    mean_throat_radius: 0.000
    std_throat_radius: 0.000


(RandomForestRegressor(n_estimators=200, random_state=42), -0.3280555195499961)